# Executor de experimentos

- Usado para a execução de cada experimento no ambiente do Google Colab.
- Cada experimento foi realizado em três rodadas com seeds diferentes
- Os resultados de cada experimento foram salvas em arquivos no Google Drive e num banco de dados

Verificação do ambiente de execução

In [1]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

!nvidia-smi

2.19.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Tue Apr 28 15:30:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P8             15W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                       

Clonagem do repositório atualizado no GitHub

In [2]:
%cd /content
!git clone https://github.com/amartinsmg/classification-of-medical-images-using-cnn.git
%cd /content/classification-of-medical-images-using-cnn

/content
fatal: destination path 'classification-of-medical-images-using-cnn' already exists and is not an empty directory.
/content/classification-of-medical-images-using-cnn


Montagem do Google Drive

In [3]:
from google.colab import drive

drive.mount("/content/drive")
BASE_PATH = "/content/drive/MyDrive/classification-of-medical-images-using-cnn/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Configuração dos parâmetros do experimento

In [4]:
EXPERIMENT_NAME = "efficientnet-baseline"
BASE_MODEL = "efficientnet"
NORMALIZATION = "preprocess-input"
DATA_AUG = True
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_WEIGHT = False
LEARNING_RATE = 0.001
EPOCHS = 10
THRESHOLD = 0.5
seeds = [42, 123, 999]

Criação da engine do banco de dados
- Foi usado um banco de dados sqlite hospedado também no Google Drive

In [5]:
from src.db import insert_run, get_engine

DB_PATH = f"sqlite:///{BASE_PATH}/experiments.db"

engine = get_engine(DB_PATH)

Realização das três rodadas do experimento com:
- Treinamento do modelo
- Teste do modelo treinado
- Persitência dos parâmetros e dos resulados da rodada em banco de dados

In [6]:
from src.train import train_pipeline
from src.test import test_pipeline

for i in range(3):
    run_id = i + 1
    config = train_pipeline(
        base_dir=BASE_PATH,
        experiment_name=EXPERIMENT_NAME,
        run_id=run_id,
        normalization=NORMALIZATION,
        base_model=BASE_MODEL,
        data_augmentation=DATA_AUG,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_weights=CLASS_WEIGHT,
        learning_rate=LEARNING_RATE,
        epochs=EPOCHS,
        seed=seeds[i],
    )

    metrics = test_pipeline(
        base_dir=BASE_PATH,
        experiment_name=EXPERIMENT_NAME,
        run_id=run_id,
        normalization=NORMALIZATION,
        base_model=BASE_MODEL,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        threshold=THRESHOLD,
    )

    insert_run(
        engine=engine,
        experiment=EXPERIMENT_NAME,
        run_name=f"run_{run_id}",
        config=config,
        metrics=metrics,
    )

Found 5216 files belonging to 2 classes.
Found 16 files belonging to 2 classes.
Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 141s 643ms/step - AUC: 0.9720 - accuracy: 0.9214 - loss: 0.1869 - val_AUC: 1.0000 - val_accuracy: 0.8750 - val_loss: 0.1844
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 18s 37ms/step - AUC: 0.9899 - accuracy: 0.9553 - loss: 0.1134 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 0.0797
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - AUC: 0.9924 - accuracy: 0.9611 - loss: 0.0937 - val_AUC: 1.0000 - val_accuracy: 0.8750 - val_loss: 0.1984
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - AUC: 0.9933 - accuracy: 0.9664 - loss: 0.0907 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 0.0774
Epoch 5/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - AUC: 0.9941 - accuracy: 0.9668 - loss: 0.0850 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 0.0527
Epoch 6/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - AUC: 0.9950 - accuracy: 0.9686 - loss: 0.0790 - val_AUC: